In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import os
import glob
from typing import Dict, Any, Optional
import plotly.graph_objects as go



# Max conductividad de wells en el tiempo

In [ ]:
def extract_max_conductivity_from_folders(
    folder_paths,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    recursive=False
):
    """
    Lee todos los archivos CSV de una lista de folders y extrae:
    - valor máximo de conductividad
    - valor correspondiente de Vertical Position m
    - ID = nombre del archivo sin extensión

    Parameters
    ----------
    folder_paths : list of str
        Lista de rutas de carpetas que contienen archivos .csv
    vp_column : str
        Nombre de la columna de posición vertical
    sec_column : str
        Nombre de la columna de conductividad eléctrica
    recursive : bool
        Si True, busca también en subcarpetas

    Returns
    -------
    pd.DataFrame
        Tabla con ID, max_conductivity_uScm, vp_at_max_m y file_path
    """
    results = []

    for folder in folder_paths:
        pattern = os.path.join(folder, '**', '*.csv') if recursive else os.path.join(folder, '*.csv')
        csv_files = glob.glob(pattern, recursive=recursive)

        for file_path in csv_files:
            try:
                df = pd.read_csv(file_path)

                if vp_column not in df.columns or sec_column not in df.columns:
                    print(f'[WARNING] Columnas no encontradas en: {file_path}')
                    continue

                # Convertir a numérico por si vienen strings
                df[vp_column] = pd.to_numeric(df[vp_column], errors='coerce')
                df[sec_column] = pd.to_numeric(df[sec_column], errors='coerce')

                # Quitar filas con NaN en columnas clave
                df_valid = df.dropna(subset=[vp_column, sec_column]).copy()

                if df_valid.empty:
                    print(f'[WARNING] Sin datos válidos en: {file_path}')
                    continue

                # Índice del máximo de conductividad
                idx_max = df_valid[sec_column].idxmax()

                max_cond = df_valid.loc[idx_max, sec_column]
                vp_at_max = df_valid.loc[idx_max, vp_column]

                file_name = os.path.basename(file_path)
                file_id = os.path.splitext(file_name)[0]

                results.append({
                    'ID': file_id,
                    'max_conductivity_uScm': max_cond,
                    'vp_at_max_m': vp_at_max,
                    'file_path': file_path
                })

            except Exception as e:
                print(f'[ERROR] No se pudo procesar {file_path}: {e}')

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values(by='max_conductivity_uScm', ascending=True).reset_index(drop=True)

    if not results_df.empty:
        results_df['max_conductivity_uScm'] = results_df['max_conductivity_uScm'].round(2)

    return results_df


# =========================
# EJEMPLO DE USO
# =========================

folder_paths = [
    r'C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_bw3',
    r'C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_lrs69',
    r'C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_lrs70'
]

df_maximos = extract_max_conductivity_from_folders(
    folder_paths=folder_paths,
    vp_column='Vertical Position m',
    sec_column='Temp °C',
    recursive=False
)


print(df_maximos)

# Guardar a CSV si quieres
# df_maximos.to_csv('max_conductivity_summary.csv', index=False)